# Sprungantworten: wie schnell reagieren Lenkung und Antrieb?

Auswertung der Aufnahmen mit `step_test_profile.py` (Profile `steering` und `speed`),
für das echte Auto und für die Simulation.

**Für jeden Sprung im Befehl** (`/ackermann_cmd`) wird gemessen, wie lange das Fahrzeug braucht, bis die Reaktion

- **10 %** (≈ Totzeit), **63 %** (≈ Zeitkonstante τ) und **90 %** des Weges geschafft hat.

**Gemessene Grösse:**
- Lenksprung: Krümmung κ = Gierrate / Geschwindigkeit
  (echt: IMU und VESC-Drehzahl, Sim: `/odom`)
- Geschwindigkeitssprung: Geschwindigkeit
  (echt: VESC-Drehzahl aus `/sensors/core`, zur Kontrolle Vicon, Sim: `/odom`)

**Zusätzlich (nur echt):**
- Verzögerung in ROS: `/ackermann_cmd` bis `/commands/servo/position` bzw. `/commands/motor/speed`
- Netzwerkverzögerung Vicon: Ankunft auf dem Raspi minus `header.stamp` (setzt synchrone Uhren voraus, chrony)

## 1. Einstellungen
In `BAGS` alle Aufnahmen eintragen, die verglichen werden sollen (echt und Sim gemischt ist möglich).

In [43]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore

DATA_DIR = Path(
    "/home/tommy/Documents/autonomous-systems/athena/athena-data-analysis/"
    "sim-to-real-gap/stage-2-testing"
)
# (label, bag folder)
BAGS = [
    ("real steering 01", DATA_DIR / "data-real-athena" / "real_steering_01"),
    ("real steering 02", DATA_DIR / "data-real-athena" / "real_steering_02"),
    ("real steering 03", DATA_DIR / "data-real-athena" / "real_steering_03"),
    ("real speed 01", DATA_DIR / "data-real-athena" / "real_speed_01"),
    ("real speed 02", DATA_DIR / "data-real-athena" / "real_speed_02"),
    ("real speed 03", DATA_DIR / "data-real-athena" / "real_speed_03"),
    ("sim steering 01", DATA_DIR / "data-sim-athena" / "sim_steering_01"),
    ("sim steering 02", DATA_DIR / "data-sim-athena" / "sim_steering_02"),
    ("sim steering 03", DATA_DIR / "data-sim-athena" / "sim_steering_03"),
    ("sim speed 01", DATA_DIR / "data-sim-athena" / "sim_speed_01"),
    ("sim speed 02", DATA_DIR / "data-sim-athena" / "sim_speed_02"),
    ("sim speed 03", DATA_DIR / "data-sim-athena" / "sim_speed_03")
]

CMD_TOPIC = "/ackermann_cmd"
SIM_ODOM_TOPIC = "/odom"
VICON_TOPIC = "/vicon/athena_with_wheel_tracker/athena_with_wheel_tracker"
IMU_TOPIC = "/sensors/imu/raw"
VESC_STATE_TOPIC = "/sensors/core"
SERVO_CMD_TOPIC = "/commands/servo/position"
MOTOR_CMD_TOPIC = "/commands/motor/speed"

SPEED_TO_ERPM_GAIN = 3793.0  # from ackermann_to_vesc: ERPM = gain * speed
IMU_UNIT = "deg/s"  # unit of the IMU angular velocity ("deg/s" or "rad/s")
IMU_SIGN = "auto"  # "auto" (from Vicon), +1 or -1

STEER_STEP_MIN = 0.01  # rad, smaller command changes are not counted as steps
SPEED_STEP_MIN = 0.05  # m/s
MIN_SPEED = 0.2  # m/s, below this the curvature is not computed
SMOOTH_S = 0.05  # s, moving average of the response signals
BASELINE_S = 0.3  # s before the step used for the start value
FINAL_FROM_S = 0.8  # s after the step: start of the window for the final value
FINAL_TO_S = 1.3  # s after the step: end of the window for the final value
# smaller total reactions are reported as "no effect" (e.g. a steering step inside the play)
MIN_CHANGE_CURVATURE = 0.03  # 1/m
MIN_CHANGE_SPEED = 0.05  # m/s

## 2. Hilfsfunktionen

In [44]:
def yaw_from_quaternion(q):
    """Return the yaw angle of a quaternion."""
    siny_cosp = 2.0 * (q.w * q.z + q.x * q.y)
    cosy_cosp = 1.0 - 2.0 * (q.y**2 + q.z**2)
    return math.atan2(siny_cosp, cosy_cosp)


def stamp_to_s(stamp):
    return stamp.sec + stamp.nanosec * 1e-9


def read_bag(bag_path):
    """Return {topic: [(bag_time_s, msg), ...]} for all topics of interest that exist."""
    wanted = [CMD_TOPIC, SIM_ODOM_TOPIC, VICON_TOPIC, IMU_TOPIC, VESC_STATE_TOPIC,
              SERVO_CMD_TOPIC, MOTOR_CMD_TOPIC]
    typestore = get_typestore(Stores.ROS2_JAZZY)
    result = {}
    with AnyReader([bag_path], default_typestore=typestore) as reader:
        connections = []
        for connection in reader.connections:
            if connection.topic in wanted and connection.msgcount > 0:
                connections.append(connection)
                result[connection.topic] = []
        for connection, timestamp, rawdata in reader.messages(connections=connections):
            msg = reader.deserialize(rawdata, connection.msgtype)
            result[connection.topic].append((timestamp / 1e9, msg))
    print(f"{bag_path.name}: " + ", ".join(f"{topic} ({len(msgs)})" for topic, msgs in result.items()))
    return result


def moving_average(values, window_samples):
    """Centered moving average: replaces each value by the mean of its neighbours."""
    window_samples = max(1, int(window_samples))  # at least 1 sample (1 = no smoothing)
    # normalized kernel: equal weights 1/N that sum to 1, so the result is the mean, not the sum
    kernel = np.ones(window_samples) / window_samples
    # repeat the first/last value at the edges so the output has the same length as the input
    # and output[i] belongs to the window centered on values[i]
    padded = np.pad(values, (window_samples // 2, window_samples - 1 - window_samples // 2), mode="edge")
    # slide the kernel over the signal and take the weighted sum at each position
    return np.convolve(padded, kernel, mode="valid")


def smooth(t, values, window_s):
    if len(t) < 3:
        return values
    rate = (len(t) - 1) / (t[-1] - t[0]) # calculate the sampling rate from the timestamps
    return moving_average(values, round(window_s * rate))


def find_steps(t_cmd, values, min_change):
    """Return a list of (time, before, after) where the command jumps by more than min_change."""
    steps = []
    for i in range(1, len(values)):
        if abs(values[i] - values[i - 1]) > min_change:
            steps.append((t_cmd[i], values[i - 1], values[i]))
    return steps


def response_times(t, y, t_step, t_next):
    """Times after t_step at which y reaches 10 / 63 / 90 % of its change, and the change itself."""
    before = (t >= t_step - BASELINE_S) & (t < t_step)
    final_end = min(t_step + FINAL_TO_S, t_next - 0.05)
    after = (t >= t_step + FINAL_FROM_S) & (t <= final_end)
    if np.sum(before) < 3 or np.sum(after) < 3:
        return None
    start_value = float(np.mean(y[before]))
    final_value = float(np.mean(y[after]))
    change = final_value - start_value
    result = {"start": start_value, "final": final_value, "change": change}
    window = (t >= t_step) & (t <= final_end)
    progress = (y[window] - start_value) / change if abs(change) > 1e-9 else np.zeros(np.sum(window))
    t_window = t[window]
    for level, name in [(0.1, "t10_ms"), (0.63, "t63_ms"), (0.9, "t90_ms")]:
        reached = np.nonzero(progress >= level)[0]
        result[name] = (t_window[reached[0]] - t_step) * 1000.0 if len(reached) > 0 else np.nan
        # dead time estimate: extend the line through the 10 % and 63 % points back to 0 %
    if not np.isnan(result["t10_ms"]) and not np.isnan(result["t63_ms"]):
        slope_ms = (result["t63_ms"] - result["t10_ms"]) / (0.63 - 0.1)  # ms per unit of progress
        result["t0_ms"] = result["t10_ms"] - 0.1 * slope_ms
    else:
        result["t0_ms"] = np.nan
    return result


def first_change_after(t, values, t_step, tolerance):
    """Delay until a command topic (e.g. /commands/servo/position) changes after t_step."""
    before = values[t < t_step]
    if len(before) == 0:
        return np.nan
    last_value = before[-1]
    later = np.nonzero((t >= t_step) & (np.abs(values - last_value) > tolerance))[0]
    if len(later) == 0:
        return np.nan
    return (t[later[0]] - t_step) * 1000.0

## 3. Signale aufbereiten

Für jede Aufnahme: Befehle, Geschwindigkeit, Gierrate und Krümmung auf einer gemeinsamen Zeitachse (Zeit der Aufnahme auf dem Raspi bzw. Sim-Zeit).
Echte Aufnahmen werden an `VICON_TOPIC` oder `IMU_TOPIC` erkannt, Sim-Aufnahmen an `/odom` ohne IMU.

In [ ]:
def prepare(label, bag_path):
    data = read_bag(bag_path)
    is_real = IMU_TOPIC in data or VICON_TOPIC in data
    signals = {"label": label, "real": is_real}

    cmd = data[CMD_TOPIC]
    t_cmd = np.array([time_s for time_s, msg in cmd])
    order = np.argsort(t_cmd)
    signals["t_cmd"] = t_cmd[order]
    signals["cmd_speed"] = np.array([msg.drive.speed for time_s, msg in cmd])[order]
    signals["cmd_steer"] = np.array([msg.drive.steering_angle for time_s, msg in cmd])[order]

    if is_real:
        # speed from the VESC motor speed (local on the Raspi, no network)
        if VESC_STATE_TOPIC in data:
            t_v = np.array([time_s for time_s, msg in data[VESC_STATE_TOPIC]])
            erpm = np.array([msg.state.speed for time_s, msg in data[VESC_STATE_TOPIC]])
            signals["t_speed"] = t_v
            signals["speed"] = smooth(t_v, np.abs(erpm) / SPEED_TO_ERPM_GAIN, SMOOTH_S) 
            signals["speed_source"] = "VESC ERPM"
        # Vicon: header stamp = time on the server when the pose was created (needs chrony)
        if VICON_TOPIC in data:
            vicon = data[VICON_TOPIC]
            t_arrival = np.array([time_s for time_s, msg in vicon]) # arrival time on the Raspi
            t_header = np.array([stamp_to_s(msg.header.stamp) for time_s, msg in vicon])
            network_ms = (t_arrival - t_header) * 1000.0
            signals["vicon_delay_ms"] = network_ms
            x = np.array([msg.pose.position.x for time_s, msg in vicon])
            y = np.array([msg.pose.position.y for time_s, msg in vicon])
            yaw = np.unwrap(np.array([yaw_from_quaternion(msg.pose.orientation) for time_s, msg in vicon]))
            order_v = np.argsort(t_header)
            t_h = t_header[order_v]
            keep = np.concatenate(([True], np.diff(t_h) > 1e-6))
            t_h = t_h[keep]
            x = moving_average(x[order_v][keep], 10)
            y = moving_average(y[order_v][keep], 10)
            yaw = yaw[order_v][keep]
            signals["t_vicon"] = t_h
            signals["vicon_speed"] = smooth(t_h, np.hypot(np.gradient(x, t_h), np.gradient(y, t_h)), SMOOTH_S) # 
            signals["vicon_yaw_rate"] = smooth(t_h, np.gradient(yaw, t_h), SMOOTH_S)
            if "speed" not in signals:
                signals["t_speed"] = t_h
                signals["speed"] = signals["vicon_speed"]
                signals["speed_source"] = "Vicon"
        # yaw rate from the IMU (local on the Raspi)
        if IMU_TOPIC in data:
            t_i = np.array([time_s for time_s, msg in data[IMU_TOPIC]])
            yaw_rate = np.array([msg.angular_velocity.z for time_s, msg in data[IMU_TOPIC]])
            if IMU_UNIT == "deg/s":
                yaw_rate = np.radians(yaw_rate)
            sign = 1.0
            if IMU_SIGN == "auto" and "t_vicon" in signals:
                # vicon system measurements at different timestamps than the IMU, so interpolate to the IMU timestamps
                vicon_at_imu = np.interp(t_i, signals["t_vicon"], signals["vicon_yaw_rate"]) 
                turning = np.abs(vicon_at_imu) > 0.2 # 0.2 rad/s = 11.5 deg/s, ignore small turns
                if np.sum(turning) > 10: # enough samples to compute a median, otherwise the sign is unreliable
                    sign = float(np.sign(np.median(yaw_rate[turning] / vicon_at_imu[turning])))
            elif IMU_SIGN != "auto":
                sign = float(IMU_SIGN)
            signals["t_yaw"] = t_i
            signals["yaw_rate"] = smooth(t_i, sign * yaw_rate, SMOOTH_S)
            signals["yaw_source"] = f"IMU (sign {sign:+.0f})"
        elif "t_vicon" in signals:
            signals["t_yaw"] = signals["t_vicon"]
            signals["yaw_rate"] = signals["vicon_yaw_rate"]
            signals["yaw_source"] = "Vicon"
        # command topics to the VESC (for the ROS delay)
        for topic, key in [(SERVO_CMD_TOPIC, "servo"), (MOTOR_CMD_TOPIC, "motor")]:
            if topic in data:
                signals[f"t_{key}"] = np.array([time_s for time_s, msg in data[topic]])
                signals[key] = np.array([msg.data for time_s, msg in data[topic]])
    else:
        odom = data[SIM_ODOM_TOPIC]
        t_o = np.array([time_s for time_s, msg in odom])
        signals["t_speed"] = t_o
        signals["speed"] = smooth(t_o, np.array([msg.twist.twist.linear.x for time_s, msg in odom]), SMOOTH_S)
        signals["speed_source"] = "/odom"
        signals["t_yaw"] = t_o
        signals["yaw_rate"] = smooth(t_o, np.array([msg.twist.twist.angular.z for time_s, msg in odom]), SMOOTH_S)
        signals["yaw_source"] = "/odom"

    # curvature on the yaw rate time base
    if "t_yaw" in signals and "t_speed" in signals:
        speed_at_yaw = np.interp(signals["t_yaw"], signals["t_speed"], signals["speed"])
        curvature = np.full(len(speed_at_yaw), np.nan)
        moving = speed_at_yaw > MIN_SPEED
        curvature[moving] = signals["yaw_rate"][moving] / speed_at_yaw[moving] # curvature = yaw_rate / speed, in 1/m
        signals["curvature"] = curvature 
    print(f"  -> {'real' if is_real else 'sim'}, speed from {signals.get('speed_source')}, "
          f"yaw rate from {signals.get('yaw_source')}")
    return signals


prepared = []
for label, bag_path in BAGS:
    prepared.append(prepare(label, bag_path))

real_steering_01: /ackermann_cmd (492), /commands/motor/speed (492), /commands/servo/position (492), /sensors/core (983), /sensors/imu/raw (983), /vicon/athena_with_wheel_tracker/athena_with_wheel_tracker (3638)
  -> real, speed from VESC ERPM, yaw rate from IMU (sign +1)
real_steering_02: /ackermann_cmd (467), /commands/motor/speed (492), /vicon/athena_with_wheel_tracker/athena_with_wheel_tracker (5197), /commands/servo/position (492), /sensors/core (1392), /sensors/imu/raw (1391)
  -> real, speed from VESC ERPM, yaw rate from IMU (sign +1)
real_steering_03: /ackermann_cmd (492), /vicon/athena_with_wheel_tracker/athena_with_wheel_tracker (3789), /commands/motor/speed (491), /commands/servo/position (491), /sensors/core (1016), /sensors/imu/raw (1016)
  -> real, speed from VESC ERPM, yaw rate from IMU (sign +1)
real_speed_01: /ackermann_cmd (392), /vicon/athena_with_wheel_tracker/athena_with_wheel_tracker (3374), /commands/motor/speed (392), /sensors/imu/raw (774), /commands/servo/posi

## 4. Sprünge finden und Reaktionszeiten messen

In [ ]:
rows = []
for signals in prepared:
    t_cmd = signals["t_cmd"]
    steer_steps = find_steps(t_cmd, signals["cmd_steer"], STEER_STEP_MIN)
    speed_steps = find_steps(t_cmd, signals["cmd_speed"], SPEED_STEP_MIN)
    all_steps = []
    for step in steer_steps:
        all_steps.append(("steering", step))
    for step in speed_steps:
        all_steps.append(("speed", step))
    step_times = sorted([step[0] for kind, step in all_steps])

    for kind, (t_step, before, after) in all_steps:
        later = [t for t in step_times if t > t_step + 1e-6]
        t_next = later[0] if later else t_step + 10.0
        row = {"bag": signals["label"], "kind": kind, "t_step_s": t_step - t_cmd[0],
               "cmd_from": before, "cmd_to": after}
        if kind == "steering":
            if "curvature" not in signals:
                continue
            valid = ~np.isnan(signals["curvature"])
            times = response_times(signals["t_yaw"][valid], signals["curvature"][valid], t_step, t_next)
            row["signal"] = "curvature [1/m]"
            if "servo" in signals:
                row["ros_delay_ms"] = first_change_after(signals["t_servo"], signals["servo"], t_step, 1e-4)
        else:
            times = response_times(signals["t_speed"], signals["speed"], t_step, t_next)
            row["signal"] = "speed [m/s]"
            if "motor" in signals:
                row["ros_delay_ms"] = first_change_after(signals["t_motor"], signals["motor"], t_step, 1.0)
        if times is None:
            continue
        min_change = MIN_CHANGE_CURVATURE if kind == "steering" else MIN_CHANGE_SPEED
        if abs(times["change"]) < min_change:
            # the vehicle did not react noticeably: times are meaningless
            times["t0_ms"] = np.nan
            times["t10_ms"] = np.nan
            times["t63_ms"] = np.nan
            times["t90_ms"] = np.nan
            row["note"] = "no effect"
        else:
            row["note"] = ""
        row.update(times)
        rows.append(row)

results = pd.DataFrame(rows)
columns = ["bag", "kind", "t_step_s", "cmd_from", "cmd_to", "signal", "start", "final", "change",
           "t0_ms", "t10_ms", "t63_ms", "t90_ms", "ros_delay_ms", "note"]
columns = [c for c in columns if c in results.columns]
pd.set_option("display.width", 200)
print(results[columns].round(3).to_string(index=False))

**Lesen:** `t10_ms` ≈ Totzeit, `t63_ms` ≈ Zeitkonstante τ, `t90_ms` ≈ "fast angekommen".
`change` zeigt, wie stark die Reaktion insgesamt war. Ist sie bei einem kleinen Lenksprung (z. B. 0 → 0.05) fast 0,
hat der Sprung das Lenkspiel nicht überwunden, und die Zeiten sind dann nicht aussagekräftig.

In [ ]:
# ---- averages per bag and kind ------------------------------------------------
summary = results.groupby(["bag", "kind"])[[c for c in ["t0_ms", "t10_ms", "t63_ms", "t90_ms", "ros_delay_ms"] if c in results.columns]].mean()
print("--- mean per bag and step kind ---")
print(summary.round(1).to_string())


# ---- mean dead time over all bags (large steps only, small steps are noise) ----
large_steering = results[(results["kind"] == "steering") & (results["change"].abs() > 0.3)]
print(f"\nmean dead time t0 steering: {large_steering['t0_ms'].mean():.1f} ms "
      f"(std {large_steering['t0_ms'].std():.1f} ms, {large_steering['t0_ms'].count()} steps)")

# ---- network delay of Vicon (real bags) ---------------------------------------
for signals in prepared:
    if "vicon_delay_ms" in signals:
        delay = signals["vicon_delay_ms"]
        print(f"\n{signals['label']}: Vicon arrival - header.stamp: median {np.median(delay):.1f} ms, "
              f"p95 {np.percentile(delay, 95):.1f} ms, min {np.min(delay):.1f} ms")
        if np.min(delay) < -1.0:
            print("  !! negative values: the clocks of server and Raspi are not synchronised (chrony)")

In [ ]:
# ---- real vs sim comparison ------------------------------------------------------
LARGE_CHANGE = 0.3  # 1/m, steering steps above this are clearly out of the noise

compare = results.copy()
compare["source"] = compare["bag"].str.split(" ").str[0]          # "real" or "sim"
compare["step"] = (compare["cmd_from"].map("{:+.2f}".format) + " -> "
                   + compare["cmd_to"].map("{:+.2f}".format))
compare["rise_ms"] = compare["t63_ms"] - compare["t0_ms"]


def side_by_side(data, index, columns):
    """Mean of each column for real and sim next to each other, plus the difference."""
    table = data.groupby(index + ["source"])[columns].mean().unstack("source")
    parts = {}
    for column in columns:
        part = table[column].reindex(columns=["real", "sim"])
        part["sim - real"] = part["sim"] - part["real"]
        parts[column] = part
    return pd.concat(parts, axis=1).round(3)


steering = compare[compare["kind"] == "steering"]
large = steering[steering["change"].abs() > LARGE_CHANGE]
speed = compare[compare["kind"] == "speed"]

# 1. timing of the large steering steps, all step types together
print("=== steering, large steps: timing [ms] ===")
timing = large.groupby("source")[["t0_ms", "t10_ms", "t63_ms", "t90_ms", "rise_ms"]].agg(["mean", "std"]).T
timing = timing.unstack(level=1).reindex(columns=["real", "sim"], level=0)
print(timing.round(1).to_string())

# 2. timing per steering step type
print("\n=== steering, large steps per step type: dead time and t63 [ms] ===")
print(side_by_side(large, ["step"], ["t0_ms", "t63_ms"]).round(1).to_string())

# 3. where the steering ends up (all steering steps, incl. the small ones)
print("\n=== steering, all steps: start and final curvature [1/m] ===")
print(side_by_side(steering, ["step"], ["start", "final"]).to_string())

# 4. drive steps
print("\n=== speed steps: timing [ms] and final speed [m/s] ===")
print(side_by_side(speed, ["step"], ["t10_ms", "t63_ms", "t90_ms", "final"]).round(2).to_string())

In [ ]:
# ---- VESC currents during the speed steps: is a current limit active? ----------
BATTERY_CURRENT_MAX = 2.5   # A, Motor Settings -> General -> Current
MOTOR_CURRENT_MAX = 15.0    # A
MOTOR_CURRENT_BRAKE = -5.0  # A

for label, bag_path in BAGS:
    if not label.startswith("real speed") or not bag_path.exists():
        continue
    data = read_bag(bag_path)
    core = data[VESC_STATE_TOPIC]
    cmd = data[CMD_TOPIC]
    t_start = cmd[0][0]
    t_core = np.array([time_s for time_s, msg in core]) - t_start
    speed = np.array([msg.state.speed for time_s, msg in core]) / SPEED_TO_ERPM_GAIN
    current_motor = np.array([msg.state.current_motor for time_s, msg in core])
    current_input = np.array([msg.state.current_input for time_s, msg in core])
    duty = np.array([msg.state.duty_cycle for time_s, msg in core])
    t_cmd = np.array([time_s for time_s, msg in cmd]) - t_start
    speed_cmd = np.array([msg.drive.speed for time_s, msg in cmd])

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    axes[0].step(t_cmd, speed_cmd, where="post", label="command")
    axes[0].plot(t_core, speed, label="VESC speed")
    axes[0].set_ylabel("speed [m/s]")
    axes[1].plot(t_core, current_motor, label="motor current")
    axes[1].plot(t_core, current_input, label="battery current")
    axes[1].axhline(BATTERY_CURRENT_MAX, color="tab:orange", linestyle="--", linewidth=1, label="battery max")
    axes[1].axhline(MOTOR_CURRENT_BRAKE, color="tab:blue", linestyle=":", linewidth=1, label="motor brake max")
    axes[1].set_ylabel("current [A]")
    axes[2].plot(t_core, duty, label="duty cycle")
    axes[2].set_ylabel("duty cycle [-]")
    axes[2].set_xlabel("time since first command [s]")
    axes[0].set_title(label)
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- check speed_to_erpm_gain with Vicon (straight driving only) ----------------
MIN_GAIN_SPEED = 0.3        # m/s, slower samples are left out (VESC speed estimate unreliable)
MAX_STRAIGHT_YAW_RATE = 0.1  # rad/s, only straight driving (no difference between body and wheel speed)

all_ratios = []
for signals in prepared:
    if not signals["real"] or "t_vicon" not in signals or "t_speed" not in signals:
        continue
    t = signals["t_speed"]
    v_vesc = signals["speed"]
    v_vicon = np.interp(t, signals["t_vicon"], signals["vicon_speed"])
    yaw_rate = np.interp(t, signals["t_yaw"], signals["yaw_rate"]) if "t_yaw" in signals else np.zeros(len(t))
    valid = (v_vicon > MIN_GAIN_SPEED) & (np.abs(yaw_rate) < MAX_STRAIGHT_YAW_RATE)
    if np.sum(valid) < 20:
        print(f"{signals['label']}: not enough straight samples")
        continue
    ratio = v_vesc[valid] / v_vicon[valid]
    all_ratios.append(ratio)
    print(f"{signals['label']}: VESC / Vicon median {np.median(ratio):.3f} "
          f"(p25 {np.percentile(ratio, 25):.3f}, p75 {np.percentile(ratio, 75):.3f}, {np.sum(valid)} samples)")

if all_ratios:
    ratio = np.median(np.concatenate(all_ratios))
    print(f"\nall bags: VESC / Vicon = {ratio:.3f}")
    print(f"speed_to_erpm_gain now {SPEED_TO_ERPM_GAIN:.0f} -> fitted {SPEED_TO_ERPM_GAIN * ratio:.0f}")

## 5. Plots: jede Sprungantwort normiert (0 = Startwert, 1 = Endwert)

In [ ]:
kinds = [kind for kind in ["steering", "speed"] if kind in set(results["kind"])]
for kind in kinds:
    subset = results[results["kind"] == kind]
    labels = list(dict.fromkeys(subset["bag"]))
    fig, axes = plt.subplots(1, len(labels), figsize=(7 * len(labels), 4.5), squeeze=False)
    for ax, label in zip(axes[0], labels):
        signals = [s for s in prepared if s["label"] == label][0]
        for _, row in subset[subset["bag"] == label].iterrows():
            t_step = row["t_step_s"] + signals["t_cmd"][0]
            if kind == "steering":
                valid = ~np.isnan(signals["curvature"])
                t = signals["t_yaw"][valid]
                y = signals["curvature"][valid]
            else:
                t = signals["t_speed"]
                y = signals["speed"]
            window = (t >= t_step - 0.3) & (t <= t_step + 1.3)
            if row["note"] == "no effect":
                continue
            progress = (y[window] - row["start"]) / row["change"]
            ax.plot((t[window] - t_step) * 1000.0, progress, linewidth=1.5,
                    label=f"{row['cmd_from']:+.2f} -> {row['cmd_to']:+.2f}")
        ax.axhline(0.63, color="0.5", linestyle="--", linewidth=1)
        ax.axhline(1.0, color="0.5", linestyle=":", linewidth=1)
        ax.axvline(0.0, color="0.3", linewidth=1)
        ax.set_ylim(-0.3, 1.4)
        ax.set_xlabel("time after the step command [ms]")
        ax.set_ylabel("normalised response")
        ax.set_title(f"{label}: {kind} steps (dashed: 63 %)")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, title="command")
    plt.tight_layout()
    plt.show()